# Algoritmos de Prim e Kruskall

## Árvores Geradoras Mínimas

Autor: Prof. Lucas Nunes Alegre

In [ ]:
from typing import Dict, List, Tuple
import heapq
import random
import time

In [2]:
# Grafos Valorados
# Dicionário mapeando cada nodo para uma lista de pares (vizinho, peso da aresta)
# Não pode haver arestas negativas!

G1 = {
    'S': [('A', 1), ('B', 3)],
    'A': [('S', 1), ('D', 5), ('C', 4)],
    'B': [('S', 3), ('D', 4), ('C', 1)],
    'C': [('B', 1), ('A', 4), ('E', 6)],
    'D': [('A', 5), ('B', 4), ('E', 2)],
    'E': [('D', 2), ('C', 6)],
    'F': [],
}

## Algoritmo de Prim

In [ ]:
def prim(graph: Dict[str, List[Tuple[str, int]]], start: str) -> List[Tuple[str, str, int]]:
    """Algoritmo de Prim para encontrar a árvore geradora mínima de um grafo valorado.

    Complexidade: O(|E| log |V|), onde |E| é o número de arestas e |V| é o número de vértices no grafo.
    
    Args:
        graph: Grafo valorado representado como um dicionário de adjacências.
        start: Nodo inicial para começar a construção da árvore geradora mínima.
    
    Returns:
        O custo total da árvore geradora mínima e uma lista de arestas (u, v, peso) que compõem a árvore geradora mínima.
    """
    mst_edges = []  # Lista para armazenar as arestas da árvore geradora mínima
    visited = {v : False for v in graph.keys()}  # Dicionário para marcar os nodos visitados
    min_heap = [(0, start, None)]  # Min-heap para selecionar a próxima aresta de menor peso
    custo = 0  # Variável para acumular o custo total da árvore geradora mínima

    while min_heap:
        weight, current_node, parent = heapq.heappop(min_heap)

        if visited[current_node]:
            continue
        
        visited[current_node] = True

        if parent is not None:
            mst_edges.append((parent, current_node, weight))
            custo += weight
        
        for neighbor, edge_weight in graph[current_node]:
            if not visited[neighbor]:
                heapq.heappush(min_heap, (edge_weight, neighbor, current_node))

    return custo, mst_edges

In [4]:
prim(G1, 'S')

(11,
 [('S', 'A', 1), ('S', 'B', 3), ('B', 'C', 1), ('B', 'D', 4), ('D', 'E', 2)])

## Estrutura Union-Find

In [ ]:
class UnionFind:
    """Estrutura de dados Union-Find (Disjoint Set Union) para auxiliar o algoritmo de Kruskal."""

    def __init__(self, elements: List[str]):
        """Inicializa a estrutura Union-Find com os elementos fornecidos.
            O(n), onde n é o número de elementos.
        """
        self.parent = {v: v for v in elements}  # Cada elemento é inicialmente seu próprio pai (representante do conjunto)
        self.rank = {v: 0 for v in elements}    # Rank para otimizar a união dos conjuntos

    def find(self, v):
        """Encontra o representante do conjunto ao qual o elemento v pertence, aplicando compressão de caminho.
            O(α(n)), onde α é a função inversa de Ackermann, que cresce muito lentamente e é praticamente constante para todos os valores de n encontrados na prática.
        """
        if self.parent[v] != v:
            self.parent[v] = self.find(self.parent[v])  # Path compression
        return self.parent[v]
    
    def union(self, u, v):
        """Une os conjuntos aos quais os elementos u e v pertencem, utilizando a técnica de união por rank.
            O(α(n)), onde α é a função inversa de Ackermann, que cresce muito lentamente e é praticamente constante para todos os valores de n encontrados na prática.
        """
        root_u = self.find(u)
        root_v = self.find(v)
        if root_u != root_v:
            if self.rank[root_u] > self.rank[root_v]:
                self.parent[root_v] = root_u
            elif self.rank[root_u] < self.rank[root_v]:
                self.parent[root_u] = root_v
            else:
                self.parent[root_v] = root_u
                self.rank[root_u] += 1


## Algoritmo de Kruskall

In [ ]:
def kruskal(graph: Dict[str, List[Tuple[str, int]]]) -> List[Tuple[str, str, int]]:
    """Algoritmo de Kruskal para encontrar a árvore geradora mínima de um grafo valorado.

    Complexidade: O(|E| log |E|) devido à ordenação das arestas, onde E é o conjunto de arestas no grafo.
    
    Args:
        graph: Grafo valorado representado como um dicionário de adjacências.
    
    Returns:
        O custo total da árvore geradora mínima e uma lista de arestas (u, v, peso) que compõem a árvore geradora mínima.
    """
    # Criar uma lista de todas as arestas do grafo
    edges = []
    for node, neighbors in graph.items():
        for neighbor, weight in neighbors:
            if (neighbor, node, weight) not in edges:  # Evitar duplicatas
                edges.append((node, neighbor, weight))
    
    # Ordenar as arestas pelo peso
    edges.sort(key=lambda x: x[2])
    
    disjoint_set = UnionFind(list(graph.keys()))
    mst_edges = []  # Lista para armazenar as arestas da árvore geradora mínima
    custo = 0  # Variável para acumular o custo total da árvore geradora mínima
    num_edges = 0  # Contador para o número de arestas adicionadas à árvore geradora mínima

    for u, v, weight in edges:
        if disjoint_set.find(u) != disjoint_set.find(v):  # Verificar se os nodos u e v estão em conjuntos disjuntos
            disjoint_set.union(u, v)
            mst_edges.append((u, v, weight))

            custo += weight
            num_edges += 1

            if num_edges == len(graph.keys()) - 1:  # A árvore geradora mínima terá exatamente |V| - 1 arestas
                break

    return custo, mst_edges

In [17]:
kruskal(G1)

(11,
 [('S', 'A', 1), ('B', 'C', 1), ('D', 'E', 2), ('S', 'B', 3), ('B', 'D', 4)])